# ForestUNet — Training Notebook (Multi-Tile)

**Run this on Kaggle (free GPU tier T4). Then download `unet_forest.pth` and drop it into `ml_models/` on your laptop.**

---

## What this notebook does

1. Reads `training_manifest.json` from your uploaded dataset — no tile IDs hardcoded
2. For each of the 6 tiles (2 tropical, 2 subtropical, 2 temperate):
   - Downloads the correct Hansen GFC label tile from Google Cloud
   - Loads the best before (2018) and after (2022) HLS scenes
   - Reprojects Hansen labels to match HLS pixel grid
   - Slices into 128×128 patches with two-stage cloud filtering
3. Trains one ForestUNet on all patches from all tiles combined
4. Evaluates and exports `unet_forest.pth`

---

## Before you start — upload your data to Kaggle

On your laptop, zip the downloaded cache folders plus the manifest:
```
cd cache/
# Windows PowerShell:
Compress-Archive -Path T36MZB,T37MBS,T43PHR,T44QLH,T55HGD,T32UPU,training_manifest.json `
                 -DestinationPath carbon-monitor-hls.zip
```

Then: kaggle.com → Datasets → New Dataset → upload zip → name it `carbon-monitor-hls`

Add it as an Input to this notebook. Files will be at `/kaggle/input/carbon-monitor-hls/`

---

## Band order (must match your pipeline exactly)

```
Index 0 = B02 = Blue
Index 1 = B03 = Green
Index 2 = B04 = Red
Index 3 = B8A = NIR
Index 4 = B11 = SWIR1
Index 5 = B12 = SWIR2
Index 6 = Fmask (QA — not passed to model, used for cloud masking only)
```

Model input: 12 bands — before (6) + after (6) stacked. Each tile's
before/after granule IDs are read from `training_manifest.json`.

In [ ]:
!pip install -q rasterio pyproj
print('Done')

## 2 — Imports and device

In [ ]:
import os, random, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: No GPU. Enable GPU in Kaggle Settings -> Accelerator -> T4 GPU.')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 3 — Configuration (reads training_manifest.json — no tile IDs hardcoded)

In [ ]:
# Paths
HLS_CACHE_ROOT = Path('/kaggle/input/carbon-monitor-hls')
PATCH_DIR      = Path('/kaggle/working/patches')
WEIGHTS_OUT    = Path('/kaggle/working/unet_forest.pth')
HANSEN_DIR     = Path('/kaggle/working/hansen')

# Read tile catalogue from manifest — years and hansen tiles per tile
MANIFEST_PATH = HLS_CACHE_ROOT / 'training_manifest.json'
with open(MANIFEST_PATH) as f:
    MANIFEST = json.load(f)

YEAR_BEFORE = MANIFEST['year_before']   # 2018
YEAR_AFTER  = MANIFEST['year_after']    # 2022
TILES       = [t for t in MANIFEST['tiles'] if t['status'] == 'complete']

print(f'Manifest loaded: {len(TILES)} complete tiles')
print(f'Year pair: {YEAR_BEFORE} → {YEAR_AFTER}')
for t in TILES:
    print(f"  {t['tile_id']:10s}  biome={t['biome']:12s}  hansen={t['hansen_tile']}")

# Bands
SPECTRAL_BANDS = ['B02', 'B03', 'B04', 'B8A', 'B11', 'B12']
QA_BAND        = 'Fmask'

# Cloud masking — Fmask bit positions
CLOUD_BIT  = 1
SHADOW_BIT = 2
SNOW_BIT   = 5
MAX_CLOUD_FRACTION = 0.30

# Patch settings
PATCH_SIZE = 128
STRIDE     = 64

# Hansen label thresholds
TREECOVER_THRESHOLD = 10
LOSS_YEAR_MIN = YEAR_BEFORE - 2000   # 2018 → 18
LOSS_YEAR_MAX = YEAR_AFTER  - 2000   # 2022 → 22

# Training
BATCH_SIZE     = 16
LR             = 1e-4
N_EPOCHS       = 30
VAL_SPLIT      = 0.15
EARLY_STOP     = 5
BCE_POS_WEIGHT = 3.0

# Hansen GFC version
HANSEN_VERSION = 'GFC-2023-v1.11'
HANSEN_BASE    = f'https://storage.googleapis.com/earthenginepartners-hansen/{HANSEN_VERSION}'

print('\nConfiguration ready')

## 4 — Per-tile loop: Hansen download → scene load → patch generation

Replaces the old single-tile cells 4–12. Iterates over all tiles in the manifest.
All patches from all tiles land in the same `PATCH_DIR` — the model trains on everything combined.

Hansen tiles are downloaded once and cached in `/kaggle/working/hansen/`.

In [ ]:
import urllib.request

_L30_BAND_MAP = {'B8A': 'B05', 'B11': 'B06', 'B12': 'B07'}

def find_band_file(scene_dir, band):
    matches = [f for f in scene_dir.glob('*.tif') if f.name.endswith(f'.{band}.tif')]
    if matches: return matches[0]
    alt = _L30_BAND_MAP.get(band)
    if alt:
        matches = [f for f in scene_dir.glob('*.tif') if f.name.endswith(f'.{alt}.tif')]
        if matches: return matches[0]
    return None

def discover_scenes(tile_root, year):
    year_dir = tile_root / str(year)
    if not year_dir.exists():
        raise FileNotFoundError(f'No scenes at {year_dir}')
    required = SPECTRAL_BANDS + [QA_BAND]
    valid = []
    for d in sorted(year_dir.iterdir()):
        if not d.is_dir(): continue
        missing = [b for b in required if find_band_file(d, b) is None]
        if missing: print(f'  Skip {d.name}: missing {missing}')
        else: valid.append(d)
    return valid

def cloud_fraction(scene_dir):
    path = find_band_file(scene_dir, QA_BAND)
    if path is None: return 1.0
    with rasterio.open(path) as s: qa = s.read(1).astype(np.uint8)
    cloud  = ((qa >> CLOUD_BIT)  & 1).astype(bool)
    shadow = ((qa >> SHADOW_BIT) & 1).astype(bool)
    snow   = ((qa >> SNOW_BIT)   & 1).astype(bool)
    return float((cloud | shadow | snow).mean())

def load_scene(scene_dir):
    bands = []
    profile = None
    for band in SPECTRAL_BANDS:
        with rasterio.open(find_band_file(scene_dir, band)) as s:
            if profile is None: profile = s.profile.copy()
            bands.append(s.read(1).astype(np.float32))
    with rasterio.open(find_band_file(scene_dir, QA_BAND)) as s:
        fmask = s.read(1).astype(np.uint8)
    return np.stack(bands, axis=0), fmask, profile

def pick_best(scenes):
    scored = sorted([(s, cloud_fraction(s)) for s in scenes], key=lambda x: x[1])
    print(f'    Best: {scored[0][0].name}  cloud={scored[0][1]:.1%}')
    return scored[0][0]

def normalise_hls(image):
    out = np.empty_like(image, dtype=np.float32)
    for i in range(image.shape[0]):
        band = np.clip(image[i].astype(np.float32), 0, 10000)
        bmin, bmax = band.min(), band.max()
        out[i] = (band - bmin) / (bmax - bmin + 1e-8)
    return out

def fmask_to_invalid(fmask):
    qa = fmask.astype(np.uint8)
    return ((qa >> CLOUD_BIT) & 1).astype(bool) | \
           ((qa >> SHADOW_BIT) & 1).astype(bool) | \
           ((qa >> SNOW_BIT) & 1).astype(bool)

def reproject_to_hls(hansen_path, reference_profile, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists():
        with rasterio.open(out_path) as s: return s.read(1)
    dst = np.zeros((1, reference_profile['height'], reference_profile['width']), dtype=np.uint8)
    with rasterio.open(hansen_path) as src:
        reproject(
            source=rasterio.band(src, 1), destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=reference_profile['transform'],
            dst_crs=reference_profile['crs'],
            resampling=Resampling.nearest,
        )
    prof = reference_profile.copy()
    prof.update({'count': 1, 'dtype': 'uint8'})
    with rasterio.open(out_path, 'w', **prof) as dst_f: dst_f.write(dst)
    return dst[0]

# ── Main loop ────────────────────────────────────────────────────────────
PATCH_DIR.mkdir(parents=True, exist_ok=True)
HANSEN_DIR.mkdir(parents=True, exist_ok=True)

total_patches = 0

for tile_info in TILES:
    tile_id     = tile_info['tile_id']
    hansen_tile = tile_info['hansen_tile']
    biome       = tile_info['biome']

    print(f'\n{"="*60}')
    print(f'Tile {tile_id}  biome={biome}  hansen={hansen_tile}')
    print(f'{"="*60}')

    # 1 — Download Hansen labels (cached — skip if already downloaded)
    hansen_files = {
        'treecover2000': HANSEN_DIR / f'Hansen_{HANSEN_VERSION}_treecover2000_{hansen_tile}.tif',
        'lossyear':      HANSEN_DIR / f'Hansen_{HANSEN_VERSION}_lossyear_{hansen_tile}.tif',
    }
    for layer, local_path in hansen_files.items():
        if local_path.exists():
            print(f'  Hansen {layer}: already downloaded')
            continue
        url = f'{HANSEN_BASE}/Hansen_{HANSEN_VERSION}_{layer}_{hansen_tile}.tif'
        print(f'  Downloading Hansen {layer} ...')
        urllib.request.urlretrieve(url, local_path)
        print(f'  Saved {local_path.name} ({local_path.stat().st_size/1e6:.1f} MB)')

    # 2 — Discover and load best scenes
    tile_root = HLS_CACHE_ROOT / tile_id
    print(f'  Loading before ({YEAR_BEFORE}):')
    before_scenes = discover_scenes(tile_root, YEAR_BEFORE)
    best_before   = pick_best(before_scenes)
    before_image, before_fmask, before_profile = load_scene(best_before)

    print(f'  Loading after ({YEAR_AFTER}):')
    after_scenes = discover_scenes(tile_root, YEAR_AFTER)
    best_after   = pick_best(after_scenes)
    after_image, after_fmask, after_profile = load_scene(best_after)

    # 3 — Normalise
    before_norm = normalise_hls(before_image)
    after_norm  = normalise_hls(after_image)

    # 4 — Cloud masks
    before_invalid   = fmask_to_invalid(before_fmask)
    after_invalid    = fmask_to_invalid(after_fmask)
    combined_invalid = before_invalid | after_invalid
    print(f'  Cloud: before={before_invalid.mean():.1%}  after={after_invalid.mean():.1%}  combined={combined_invalid.mean():.1%}')

    # 5 — Reproject Hansen to HLS grid
    reproj_dir = Path(f'/kaggle/working/hansen_reprojected/{tile_id}')
    treecover  = reproject_to_hls(hansen_files['treecover2000'], before_profile,
                                  reproj_dir / 'treecover2000.tif')
    lossyear   = reproject_to_hls(hansen_files['lossyear'],      before_profile,
                                  reproj_dir / 'lossyear.tif')

    # 6 — Build loss label
    forest_2000    = treecover > TREECOVER_THRESHOLD
    loss_in_period = (lossyear >= LOSS_YEAR_MIN) & (lossyear <= LOSS_YEAR_MAX) & (lossyear > 0)
    loss_label     = (forest_2000 & loss_in_period).astype(np.uint8)
    print(f'  Forest 2000: {forest_2000.mean():.1%}  Loss {YEAR_BEFORE}-{YEAR_AFTER}: {loss_label.mean():.3%}')

    if loss_label.mean() < 0.001:
        print(f'  WARNING: Very few loss pixels for {tile_id} — patches may be sparse')

    # 7 — Slice into patches (prefix filename with tile_id to avoid collision)
    H, W = before_norm.shape[1], before_norm.shape[2]
    n_saved = n_cloudy = n_no_forest = 0

    for row in range(0, H - PATCH_SIZE + 1, STRIDE):
        for col in range(0, W - PATCH_SIZE + 1, STRIDE):
            re, ce       = row + PATCH_SIZE, col + PATCH_SIZE
            cloud_patch  = combined_invalid[row:re, col:ce]

            if cloud_patch.mean() > MAX_CLOUD_FRACTION:
                n_cloudy += 1; continue
            if not forest_2000[row:re, col:ce].any():
                n_no_forest += 1; continue

            stacked    = np.concatenate([before_norm[:, row:re, col:ce],
                                         after_norm[:,  row:re, col:ce]], axis=0)  # (12,128,128)
            label_patch = loss_label[row:re, col:ce]
            valid_mask  = ~cloud_patch

            np.savez_compressed(
                PATCH_DIR / f'{tile_id}_patch_{row:05d}_{col:05d}.npz',
                image=stacked, label=label_patch, valid_mask=valid_mask,
            )
            n_saved += 1

    total_patches += n_saved
    print(f'  Patches: saved={n_saved}  cloudy_skipped={n_cloudy}  no_forest_skipped={n_no_forest}')

print(f'\nAll tiles done. Total patches: {total_patches}')
if total_patches < 200:
    print('WARNING: Low patch count. Check cloud fractions and loss label densities above.')

## 13 — Dataset and DataLoader

In [ ]:
class ForestPatchDataset(Dataset):
    def __init__(self, patch_dir, augment=False):
        self.files   = sorted(patch_dir.glob('*.npz'))
        self.augment = augment
        if not self.files: raise FileNotFoundError(f'No patches in {patch_dir}')
        print(f'Dataset: {len(self.files)} patches')

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        img   = torch.from_numpy(d['image']).float()       # (12,128,128)
        label = torch.from_numpy(d['label']).float()       # (128,128)
        valid = torch.from_numpy(d['valid_mask']).float()  # (128,128)
        if self.augment:
            if random.random() > 0.5:
                img   = torch.flip(img,   [2])
                label = torch.flip(label, [1])
                valid = torch.flip(valid, [1])
            if random.random() > 0.5:
                img   = torch.flip(img,   [1])
                label = torch.flip(label, [0])
                valid = torch.flip(valid, [0])
        return img, label, valid

full_ds  = ForestPatchDataset(PATCH_DIR, augment=True)
val_size = max(1, int(len(full_ds) * VAL_SPLIT))
trn_size = len(full_ds) - val_size
trn_ds, val_ds = random_split(full_ds, [trn_size, val_size],
                               generator=torch.Generator().manual_seed(SEED))
trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {trn_size}  Val: {val_size}')

## 14 — Model (12-band input)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(i,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.b(x)

class Down(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(nn.MaxPool2d(2), DoubleConv(i,o))
    def forward(self, x): return self.b(x)

class Up(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.c  = DoubleConv(i, o)
    def forward(self, x, skip):
        x = self.up(x)
        dh=skip.size(2)-x.size(2); dw=skip.size(3)-x.size(3)
        x = nn.functional.pad(x,[dw//2,dw-dw//2,dh//2,dh-dh//2])
        return self.c(torch.cat([skip,x],dim=1))

class ForestUNet(nn.Module):
    def __init__(self, in_channels=12, f=64):
        super().__init__()
        self.inc   = DoubleConv(in_channels, f)
        self.d1    = Down(f,    f*2)
        self.d2    = Down(f*2,  f*4)
        self.d3    = Down(f*4,  f*8)
        self.d4    = Down(f*8,  f*16)
        self.u1    = Up(f*16+f*8, f*8)
        self.u2    = Up(f*8+f*4,  f*4)
        self.u3    = Up(f*4+f*2,  f*2)
        self.u4    = Up(f*2+f,    f)
        self.out   = nn.Conv2d(f, 1, 1)
    def forward(self, x):
        x1=self.inc(x); x2=self.d1(x1); x3=self.d2(x2); x4=self.d3(x3); x5=self.d4(x4)
        x=self.u1(x5,x4); x=self.u2(x,x3); x=self.u3(x,x2); x=self.u4(x,x1)
        return self.out(x)

model = ForestUNet(in_channels=12).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

## 15 — Cloud-aware loss functions

Cloudy pixels get weight=0 in both BCE and Dice. They contribute nothing to the gradient.

In [ ]:
pos_w = torch.tensor([BCE_POS_WEIGHT]).to(DEVICE)

def masked_bce(logits, targets, valid):
    raw = nn.functional.binary_cross_entropy_with_logits(
        logits.squeeze(1), targets,
        pos_weight=pos_w.expand_as(logits.squeeze(1)), reduction='none')
    return (raw * valid).sum() / valid.sum().clamp(min=1)

def masked_dice(logits, targets, valid, smooth=1.0):
    p = logits.sigmoid().squeeze(1)
    pv, tv = p*valid, targets*valid
    inter = (pv*tv).sum(dim=(1,2))
    union = pv.sum(dim=(1,2)) + tv.sum(dim=(1,2))
    return 1 - ((2*inter+smooth)/(union+smooth)).mean()

def loss_fn(logits, targets, valid):
    return 0.5*masked_bce(logits,targets,valid) + 0.5*masked_dice(logits,targets,valid)

def compute_iou(logits, targets, valid, thr=0.5):
    pred=( logits.sigmoid().squeeze(1)>thr); tgt=targets.bool(); v=valid.bool()
    tp=(pred& tgt& v).sum().item(); fp=(pred&~tgt& v).sum().item(); fn=(~pred& tgt& v).sum().item()
    return tp/(tp+fp+fn+1e-8)

print('Loss functions ready')

## 16 — Training loop

In [ ]:
history={'trn_loss':[],'val_loss':[],'val_iou':[]}
best_val=float('inf'); patience=0

for epoch in range(1, N_EPOCHS+1):
    model.train(); tl=0
    for imgs,labels,valid in trn_loader:
        imgs,labels,valid=imgs.to(DEVICE),labels.to(DEVICE),valid.to(DEVICE)
        optimizer.zero_grad()
        loss=loss_fn(model(imgs),labels,valid)
        loss.backward(); optimizer.step(); tl+=loss.item()

    model.eval(); vl=vi=0
    with torch.no_grad():
        for imgs,labels,valid in val_loader:
            imgs,labels,valid=imgs.to(DEVICE),labels.to(DEVICE),valid.to(DEVICE)
            out=model(imgs); vl+=loss_fn(out,labels,valid).item(); vi+=compute_iou(out,labels,valid)

    tl/=len(trn_loader); vl/=len(val_loader); vi/=len(val_loader)
    scheduler.step()
    history['trn_loss'].append(tl); history['val_loss'].append(vl); history['val_iou'].append(vi)
    print(f'Epoch {epoch:3d}/{N_EPOCHS}  trn={tl:.4f}  val={vl:.4f}  iou={vi:.4f}')

    if vl < best_val:
        best_val=vl; patience=0
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                    'in_channels':12,'val_loss':vl,'val_iou':vi}, WEIGHTS_OUT)
        print('  Saved best model')
    else:
        patience+=1
        if patience>=EARLY_STOP: print(f'  Early stop at epoch {epoch}'); break

print(f'Done. Best val={best_val:.4f}  Weights: {WEIGHTS_OUT}')

## 17 — Training curves

In [ ]:
ep=range(1,len(history['trn_loss'])+1)
fig,(a1,a2)=plt.subplots(1,2,figsize=(12,4))
a1.plot(ep,history['trn_loss'],label='Train'); a1.plot(ep,history['val_loss'],label='Val')
a1.set_title('Loss'); a1.legend()
a2.plot(ep,history['val_iou'],color='green'); a2.set_title('Val IoU (cloud-free pixels)')
plt.tight_layout(); plt.savefig('/kaggle/working/training_curves.png',dpi=150); plt.show()

## 18 — Confusion matrix on validation set (cloud-free pixels only)

In [ ]:
ckpt=torch.load(WEIGHTS_OUT,map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print(f'Epoch {ckpt["epoch"]}  val_iou={ckpt["val_iou"]:.4f}')

tp=fp=fn=tn=0
with torch.no_grad():
    for imgs,labels,valid in val_loader:
        out=model(imgs.to(DEVICE)).cpu()
        pred=(out.sigmoid().squeeze(1)>0.5).bool()
        tgt=labels.bool(); v=valid.bool()
        tp+=(pred& tgt& v).sum().item()
        fp+=(pred&~tgt& v).sum().item()
        fn+=(~pred& tgt& v).sum().item()
        tn+=(~pred&~tgt& v).sum().item()

total=tp+fp+fn+tn
acc=  (tp+tn)/total    if total    >0 else 0
prec= tp/(tp+fp)       if tp+fp    >0 else 0
rec=  tp/(tp+fn)       if tp+fn    >0 else 0
f1=   2*prec*rec/(prec+rec+1e-8)
iou=  tp/(tp+fp+fn)    if tp+fp+fn >0 else 0

print('\n── Confusion Matrix (cloud-free pixels) ──────────────')
print(f'           Predicted+   Predicted-')
print(f'Actual+    {tp:>10,}   {fn:>10,}   <- forest loss')
print(f'Actual-    {fp:>10,}   {tn:>10,}   <- no loss')
print('\n── Metrics ──────────────────────────────────────────')
print(f'  Accuracy : {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall   : {rec:.4f}')
print(f'  F1       : {f1:.4f}')
print(f'  IoU      : {iou:.4f}')
print(f'  Total valid pixels: {total:,}')

## 19 — Visual check

In [ ]:
imgs,labels,valid=next(iter(val_loader))
with torch.no_grad(): out=model(imgs.to(DEVICE)).cpu()
preds=(out.sigmoid().squeeze(1)>0.5).numpy()

n=min(4,len(imgs))
fig,axes=plt.subplots(n,4,figsize=(14,3.5*n))
if n==1: axes=axes[np.newaxis,:]
for i in range(n):
    # Before and After RGB (channels 0-2 and 6-8 respectively)
    rb=np.clip(imgs[i][[2,1,0]].permute(1,2,0).numpy()*3,0,1)
    ra=np.clip(imgs[i][[8,7,6]].permute(1,2,0).numpy()*3,0,1)
    axes[i,0].imshow(rb);                    axes[i,0].set_title('Before RGB')
    axes[i,1].imshow(ra);                    axes[i,1].set_title('After RGB')
    axes[i,2].imshow(labels[i],cmap='Greens'); axes[i,2].set_title('Hansen label')
    axes[i,3].imshow(preds[i], cmap='Reds');   axes[i,3].set_title('U-Net prediction')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/sample_predictions.png',dpi=100)
plt.show()

## 20 — After downloading: changes needed in your repo

Because the model takes `(12, 128, 128)` input (before + after stacked),
update these two files on your laptop:

**`models/unet.py`** — change default `in_channels=6` to `in_channels=12`

**`src/prithvi.py`** — in `run_prithvi_inference()`, stack before+after:
```python
stacked = np.concatenate([patch['before'], patch['after']], axis=0)  # (12,128,128)
tensor  = torch.from_numpy(stacked).float().unsqueeze(0)             # (1,12,128,128)
```

Nothing else changes. `pipeline.py` is untouched.

---

## 21 — Download from Kaggle Output

Kaggle sidebar → **Output** → download:
- `unet_forest.pth` (~50 MB) → copy to `ml_models/unet_forest.pth`
- `training_curves.png`
- `sample_predictions.png`

```bash
# Verify on your laptop:
python -c "
from src.prithvi import load_prithvi_model
m, c = load_prithvi_model()
print(c)  # weights_loaded should be True
"
```